In [ ]:
# CODE BLOCK 2: MODEL 1 - YOLOv5

import os
import torch

# --- 1. Setup YOLOv5 ---
print("Setting up YOLOv5...")
if not os.path.exists('yolov5'):
    !git clone https://github.com/ultralytics/yolov5
%cd /content/yolov5
!pip install -qr requirements.txt  # install dependencies
from yolov5 import utils
display = utils.notebook_init()  # check
print("Setup complete.")

# --- 2. Define Dataset Path ---
# The data is already processed in our Colab session
DATA_YAML_PATH = '/content/trash_dataset_yolo/dataset.yaml'
if not os.path.exists(DATA_YAML_PATH):
    print("ERROR: dataset.yaml not found!")
else:
    print(f"Dataset found at {DATA_YAML_PATH}")

# --- 3. Train the Model ---
print("Starting YOLOv5 training... This will take a while.")
# We use yolov5s (small) as a strong baseline.
# --img 640 matches the base paper[cite: 314].
# --epochs 100 for a complete training.
# --name will create a results folder: 'runs/train/yolov5s_trash_run'
!python train.py --img 640 --batch 16 --epochs 100 --data {DATA_YAML_PATH} --weights yolov5s.pt --name yolov5s_trash_run

print("YOLOv5 Training complete.")

# --- 4. Test / Validate the Model ---
print("\nValidating YOLOv5 model to get all metrics...")
# This command runs validation on the 'val' set using the best saved model.
# This will output the full metrics table: mAP@0.5, APs, APm, APl
!python val.py --weights runs/train/yolov5s_trash_run/weights/best.pt --data {DATA_YAML_PATH} --img 640 --task val

print("\n--- YOLOv5 VALIDATION COMPLETE ---")
print("Find your training logs, weights, and validation results in:")
print("/content/yolov5/runs/train/yolov5s_trash_run/")
print("\n**SAVE THE METRICS TABLE FROM THE CONSOLE OUTPUT ABOVE FOR YOUR REPORT!**")

# Go back to the main content directory for the next model
%cd /content/

YOLOv5 🚀 v7.0-443-gbe00b6b6 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)


Streaming output truncated to the last 5000 lines.
      78/99      3.56G    0.03233   0.008612          0         86        640:  11% 12/111 [00:04<00:49,  2.01it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      78/99      3.56G    0.03205   0.008627          0         62        640:  12% 13/111 [00:05<00:44,  2.19it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      78/99      3.56G    0.03225   0.008649          0         65        640:  13% 14/111 [00:05<00:52,  1.86it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      78/99      3.56G     0.0318 

In [ ]:
# This zips the main training run folder
!zip -r /content/yolov5_results.zip /content/yolov5/runs/train/yolov5s_trash_run

# This zips the separate validation run folder (just in case)
!zip -r /content/yolov5_val_results.zip /content/yolov5/runs/val/exp

  adding: content/yolov5/runs/train/yolov5s_trash_run/ (stored 0%)
  adding: content/yolov5/runs/train/yolov5s_trash_run/P_curve.png (deflated 18%)
  adding: content/yolov5/runs/train/yolov5s_trash_run/confusion_matrix.png (deflated 36%)
  adding: content/yolov5/runs/train/yolov5s_trash_run/val_batch0_pred.jpg (deflated 13%)
  adding: content/yolov5/runs/train/yolov5s_trash_run/val_batch1_labels.jpg (deflated 12%)
  adding: content/yolov5/runs/train/yolov5s_trash_run/train_batch1.jpg (deflated 9%)
  adding: content/yolov5/runs/train/yolov5s_trash_run/labels.jpg (deflated 35%)
  adding: content/yolov5/runs/train/yolov5s_trash_run/F1_curve.png (deflated 17%)
  adding: content/yolov5/runs/train/yolov5s_trash_run/labels_correlogram.jpg (deflated 35%)
  adding: content/yolov5/runs/train/yolov5s_trash_run/train_batch0.jpg (deflated 9%)
  adding: content/yolov5/runs/train/yolov5s_trash_run/events.out.tfevents.1761316558.84a4b5132c5f.11675.0 (deflated 42%)
  adding: content/yolov5/runs/train/y

In [34]:
# CODE BLOCK 2: Train Model 1 - YOLOv8 (on RGB Data - Local Colab)

import os
import zipfile

# --- 1. Install Ultralytics ---
!pip install ultralytics -q
from ultralytics import YOLO

# --- 2. Define Dataset Path ---
DATASET_DIR = '/content/trash_dataset_yolo'
DATA_YAML_PATH = os.path.join(DATASET_DIR, 'dataset.yaml')
PROCESSED_ZIP_PATH = '/content/trash_dataset_yolo_processed.zip'

# Check if data exists, unzip from local archive if necessary
if not os.path.exists(DATA_YAML_PATH):
    print("Processed folder not found. Checking for local zip...")
    if not os.path.exists(PROCESSED_ZIP_PATH):
        print(f"ERROR: Cannot find {PROCESSED_ZIP_PATH}")
        print("Please run the preprocessing script first.")
    else:
        print(f"Found {PROCESSED_ZIP_PATH}. Unzipping...")
        with zipfile.ZipFile(PROCESSED_ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall('/content/')
        print(f"RGB dataset is ready at {DATA_YAML_PATH}")
else:
     print(f"RGB dataset is ready at {DATA_YAML_PATH}")

# --- 3. (FIX) Remove the corrupt file identified earlier ---
# It's good practice to remove it even if YOLOv5 handled it.
print("Removing known corrupt data file (if it exists)...")
corrupt_img = '/content/trash_dataset_yolo/images/val/5_1604480596098.jpg'
corrupt_label = '/content/trash_dataset_yolo/labels/val/5_1604480596098.txt'
if os.path.exists(corrupt_img): os.remove(corrupt_img); print("Removed corrupt image.")
if os.path.exists(corrupt_label): os.remove(corrupt_label); print("Removed corrupt label.")

# --- 4. Train the Model ---
if os.path.exists(DATA_YAML_PATH): # Proceed only if data is ready
    print("\nStarting YOLOv8 (RGB-Only) training...")
    model = YOLO('yolov8s.pt') # 's' for small

    results = model.train(
        data=DATA_YAML_PATH,
        imgsz=640,
        epochs=100,
        batch=16,
        name='yolov8_rgb_run'
    )
    print("YOLOv8 (RGB-Only) Training complete.")

    # --- 5. Test / Validate the Model ---
    print("\nValidating YOLOv8 (RGB-Only) model...")
    best_model_path = 'runs/detect/yolov8_rgb_run/weights/best.pt'
    if os.path.exists(best_model_path):
        model = YOLO(best_model_path)
        metrics = model.val(split='val') # Use default split from YAML
        print("\n--- YOLOv8 (RGB-Only) VALIDATION COMPLETE ---")
        print("\n**SAVE THE METRICS TABLE FROM THE CONSOLE OUTPUT!**")

        # --- 6. Save Results Locally ---
        print("\nZipping results...")
        !zip -r /content/yolov8_rgb_results.zip /content/runs/detect/yolov8_rgb_run
        print("\n--- All Done! ---")
        print("YOLOv8 (RGB-Only) results saved to /content/yolov8_rgb_results.zip")
        print("You can download this file from the 'Files' tab.")
    else:
        print(f"ERROR: Trained model not found at {best_model_path}")
else:
    print("Skipping training as dataset was not found.")

RGB dataset is ready at /content/trash_dataset_yolo/dataset.yaml
Removing known corrupt data file (if it exists)...

Starting YOLOv8 (RGB-Only) training...
Ultralytics 8.3.221 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/trash_dataset_yolo/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosai

In [35]:
# CODE BLOCK 3: TESTING the YOLOv8 RGB Model

import os
!pip install ultralytics -q # Ensure ultralytics is installed
from ultralytics import YOLO

print("\n--- Testing YOLOv8 RGB Model ---")

# --- 1. Define Paths ---
best_model_path = 'runs/detect/yolov8_rgb_run/weights/best.pt'
DATA_YAML_PATH = '/content/trash_dataset_yolo/dataset.yaml' # Path to your dataset config

# --- 2. Check if Model and Data Exist ---
if not os.path.exists(best_model_path):
    print(f"ERROR: Trained model not found at {best_model_path}")
    print("Please make sure the training completed successfully and the path is correct.")
elif not os.path.exists(DATA_YAML_PATH):
    print(f"ERROR: Dataset YAML not found at {DATA_YAML_PATH}")
    print("Please ensure the preprocessing script ran correctly.")
else:
    # --- 3. Load the Best Model ---
    print(f"Loading best model from {best_model_path}...")
    model = YOLO(best_model_path)

    # --- 4. Run Validation (Test) ---
    print("\nRunning validation on the test (validation) set...")
    metrics = model.val(
        data=DATA_YAML_PATH, # Specify the dataset config
        split='val',         # Explicitly use the validation split defined in the YAML
        batch=16,            # Use a reasonable batch size
        imgsz=640            # Use the same image size as training
    )

    print("\n--- TEST RESULTS ---")
    # The metrics object contains detailed results.
    # We can print the key mAP scores:
    print(f"mAP50-95: {metrics.box.map:.4f}") # Mean Average Precision @ IoU=0.50:0.95
    print(f"mAP50:    {metrics.box.map50:.4f}") # Mean Average Precision @ IoU=0.50
    print(f"mAP75:    {metrics.box.map75:.4f}") # Mean Average Precision @ IoU=0.75
    print("\nDetailed metrics table should be printed above.")
    print("--------------------")

    # --- (Optional) Predict on a few images for visual inspection ---
    # print("\nRunning prediction on a sample validation image...")
    # You can list images in the val folder: !ls /content/trash_dataset_yolo/images/val/ | head -n 1
    # sample_image = '/content/trash_dataset_yolo/images/val/YOUR_IMAGE_NAME.jpg'
    # if os.path.exists(sample_image):
    #     results = model.predict(source=sample_image, save=True, imgsz=640, conf=0.5)
    #     print(f"Prediction results saved in 'runs/detect/predict...'")
    # else:
    #     print(f"Sample image {sample_image} not found, skipping prediction example.")


--- Testing YOLOv8 RGB Model ---
Loading best model from runs/detect/yolov8_rgb_run/weights/best.pt...

Running validation on the test (validation) set...
Ultralytics 8.3.221 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2392.5±488.3 MB/s, size: 193.5 KB)
val: Scanning /content/trash_dataset_yolo/labels/val.cache... 446 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 446/446 491.2Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 28/28 4.2it/s 6.6s
                   all        446       1022      0.941      0.932       0.96      0.601
Speed: 1.4ms preprocess, 5.1ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /content/runs/detect/val3

--- TEST RESULTS ---
mAP50-95: 0.6015
mAP50:    0.9602
mAP75:    0.6694

Detailed metrics table should be prin

In [37]:
# Zip the YOLOv8 results folder
# Make sure the folder name 'yolov8_rgb_run' matches the 'name' argument you used in model.train()
print("Zipping YOLOv8 results...")
!zip -r /content/yolov8_rgb_results.zip /content/runs/detect/yolov8_rgb_run

print("\n--- Zipping Complete! ---")
print("Your YOLOv8 results are now saved to /content/yolov8_rgb_results.zip")
print("You can download this file.")

Zipping YOLOv8 results...
updating: content/runs/detect/yolov8_rgb_run/ (stored 0%)
updating: content/runs/detect/yolov8_rgb_run/val_batch0_labels.jpg (deflated 11%)
updating: content/runs/detect/yolov8_rgb_run/weights/ (stored 0%)
updating: content/runs/detect/yolov8_rgb_run/weights/best.pt (deflated 8%)
updating: content/runs/detect/yolov8_rgb_run/weights/last.pt (deflated 8%)
updating: content/runs/detect/yolov8_rgb_run/BoxPR_curve.png (deflated 23%)
updating: content/runs/detect/yolov8_rgb_run/val_batch2_pred.jpg (deflated 11%)
updating: content/runs/detect/yolov8_rgb_run/val_batch0_pred.jpg (deflated 12%)
updating: content/runs/detect/yolov8_rgb_run/val_batch2_labels.jpg (deflated 11%)
updating: content/runs/detect/yolov8_rgb_run/results.png (deflated 8%)
updating: content/runs/detect/yolov8_rgb_run/labels.jpg (deflated 40%)
updating: content/runs/detect/yolov8_rgb_run/BoxP_curve.png (deflated 19%)
updating: content/runs/detect/yolov8_rgb_run/results.csv (deflated 62%)
updating: c